# 09 — Extended Comparative Metrics (Imbalance-Robust)

Standalone metric add-on for the 4-model comparison (BERT, RoBERTa, DistilBERT, Detoxify).
Adds **imbalance-robust** metrics on top of NB06:

- **Sentiment (3-class):** balanced accuracy (macro recall) + MCC, with bootstrap 95% CI.
- **Toxicity (multi-label):** MCC (per-label / macro / micro) + PR-AUC (per-label / macro / micro).

Reads ONLY cached artifacts — `data/gold/test.csv` + `data/inference/<model>_<task>/*.parquet`.
**No re-training / re-inference / GPU.** Does not modify NB06 or its outputs.

Outputs:
- `reports/eval_imbalance_sentiment.csv`
- `reports/eval_imbalance_toxicity.csv`
- `reports/comparison_metrics_extended.md`

In [1]:
# Sel 1: Setup
import sys
from pathlib import Path
_here = Path.cwd()
_root = next((p for p in [_here, *_here.parents] if (p / 'src').is_dir() and (p / 'configs').is_dir()), _here)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
import os
os.chdir(_root)

import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score, matthews_corrcoef, average_precision_score

from src.runtime import load_config, print_banner, RunLog
from src.eval.bootstrap_ci import bootstrap_ci

config = load_config('configs/experiment.yaml')
print_banner('09_imbalance_metrics', config)
run_log = RunLog(notebook='09_imbalance_metrics', config_path='configs/experiment.yaml')

GOLD_ROOT = Path(config['data']['gold_root'])
INF_ROOT = Path(config['data']['inference_root'])
REPORTS = Path('reports'); REPORTS.mkdir(exist_ok=True)
SENT_LABELS = config['labels']['sentiment_classes']
TOX_LABELS = config['labels']['toxicity_labels']
MODELS = ['bert', 'roberta', 'distilbert']
BOOT_N = int(config['evaluation']['bootstrap_resamples'])
BOOT_SEED = int(config['seed'])

c:\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Notebook: 09_imbalance_metrics
Experiment: thesis-sentiment-toxicity-dota2-decade
Seed: 42  |  Git: da158c0
Started at: 2026-06-10T07:16:26+00:00Z
Versi paket:
  - python: 3.10.0
  - transformers: 5.5.0
  - torch: 2.8.0+cu128
  - datasets: 4.8.5
  - scikit-learn: 1.7.1
  - pandas: 2.3.3
  - numpy: 1.26.4


In [2]:
# Sel 2: Load gold-test + helper baca inferensi tersimpan (subset baris test)
test = pd.read_csv(GOLD_ROOT / 'test.csv')
test['key_tuple'] = list(zip(test['match_id'], test['time'], test['player_slot']))
test_keys = set(test['key_tuple'])
print(f'Gold test: {len(test):,} baris — pakai inferensi tersimpan (tanpa GPU, tanpa run ulang NB01/04/05)')

def load_inference_for_test(folder: str):
    root = INF_ROOT / folder
    if not root.exists():
        return None
    frames = [pd.read_parquet(p) for p in sorted(root.glob('*.parquet'))]
    if not frames:
        return None
    inf = pd.concat(frames, ignore_index=True)
    inf['key_tuple'] = list(zip(inf['match_id'], inf['time'], inf['player_slot']))
    return inf[inf['key_tuple'].isin(test_keys)].drop_duplicates('key_tuple', keep='first')

Gold test: 2,392 baris — pakai inferensi tersimpan (tanpa GPU, tanpa run ulang NB01/04/05)


In [3]:
# Sel 3: Sentimen 3-class — balanced accuracy + MCC (bootstrap CI 95%)
l2i = {n: i for i, n in enumerate(SENT_LABELS)}
y_true = np.array([l2i.get(s, -1) for s in test['sentiment']])

rows = []
for mk in MODELS:
    sub = load_inference_for_test(f'{mk}_sentiment')
    if sub is None:
        run_log.add_warning(f'Skip {mk}_sentiment — inferensi tidak ada'); continue
    m = test.merge(sub, on='key_tuple', how='left')
    yp = m['predicted_label'].map(l2i).fillna(-1).astype(int).values
    mask = (y_true >= 0) & (yp >= 0); a, b = y_true[mask], yp[mask]
    _, bacc_lo, bacc_hi = bootstrap_ci(balanced_accuracy_score, a, b, n_resamples=BOOT_N, seed=BOOT_SEED)
    _, mcc_lo, mcc_hi = bootstrap_ci(matthews_corrcoef, a, b, n_resamples=BOOT_N, seed=BOOT_SEED)
    rows.append({'model': mk, 'n_test': int(mask.sum()),
                 'balanced_accuracy': float(balanced_accuracy_score(a, b)),
                 'balanced_acc_ci_lo': bacc_lo, 'balanced_acc_ci_hi': bacc_hi,
                 'mcc': float(matthews_corrcoef(a, b)), 'mcc_ci_lo': mcc_lo, 'mcc_ci_hi': mcc_hi})

# Detoxify zero-shot: max_toxicity_prob>=0.5 -> negative, else neutral
sub = load_inference_for_test('detoxify_toxicity')
if sub is not None:
    m = test.merge(sub, on='key_tuple', how='left')
    yp = np.where(m['max_toxicity_prob'].fillna(0) >= 0.5, l2i['negative'], l2i['neutral'])
    mask = y_true >= 0; a, b = y_true[mask], yp[mask]
    rows.append({'model': 'detoxify_zero_shot', 'n_test': int(mask.sum()),
                 'balanced_accuracy': float(balanced_accuracy_score(a, b)),
                 'balanced_acc_ci_lo': float('nan'), 'balanced_acc_ci_hi': float('nan'),
                 'mcc': float(matthews_corrcoef(a, b)), 'mcc_ci_lo': float('nan'), 'mcc_ci_hi': float('nan')})

imb_sent = pd.DataFrame(rows)
imb_sent.to_csv(REPORTS / 'eval_imbalance_sentiment.csv', index=False)
run_log.add_output(REPORTS / 'eval_imbalance_sentiment.csv')
print(imb_sent.to_string(index=False))

             model  n_test  balanced_accuracy  balanced_acc_ci_lo  balanced_acc_ci_hi       mcc  mcc_ci_lo  mcc_ci_hi
              bert    2392           0.315051            0.298871            0.339647 -0.127197  -0.161404  -0.093223
           roberta    2392           0.342617            0.308876            0.383483 -0.087738  -0.121156  -0.053800
        distilbert    2392           0.331024            0.305965            0.363178 -0.097572  -0.131151  -0.062018
detoxify_zero_shot    2392           0.373365                 NaN                 NaN  0.007822        NaN        NaN


In [4]:
# Sel 4: Toksisitas multi-label — MCC (per-label/macro/micro) + PR-AUC (per-label/macro/micro)
y_tox = test[[f'tox_{l}' for l in TOX_LABELS]].astype(int).values

def _mcc_macro(yt, yp):
    return float(np.mean([matthews_corrcoef(yt[:, i], yp[:, i]) for i in range(yt.shape[1])]))

rows = []
for mk in MODELS + ['detoxify']:
    sub = load_inference_for_test(f'{mk}_toxicity')
    if sub is None:
        run_log.add_warning(f'Skip {mk}_toxicity — inferensi tidak ada'); continue
    m = test.merge(sub, on='key_tuple', how='left')
    yp = m[[f'pred_{l}' for l in TOX_LABELS]].fillna(0).astype(int).values
    ys = m[[f'prob_{l}' for l in TOX_LABELS]].fillna(0.0).values
    # per-label PR-AUC; NaN bila label tak punya positif di gold-test (dikecualikan dari macro)
    ap = []
    for i in range(len(TOX_LABELS)):
        if int(y_tox[:, i].sum()) == 0:
            ap.append(float('nan')); continue
        try:
            ap.append(float(average_precision_score(y_tox[:, i], ys[:, i])))
        except ValueError:
            ap.append(float('nan'))
    pr_macro = float(np.nanmean(ap)) if not np.all(np.isnan(ap)) else float('nan')
    pr_micro = float(average_precision_score(y_tox.ravel(), ys.ravel()))
    mcc_pl = [float(matthews_corrcoef(y_tox[:, i], yp[:, i])) for i in range(len(TOX_LABELS))]
    _, cmlo, cmhi = bootstrap_ci(_mcc_macro, y_tox, yp, n_resamples=BOOT_N, seed=BOOT_SEED)
    row = {'model': mk, 'n_test': len(m),
           'mcc_macro': float(np.mean(mcc_pl)), 'mcc_macro_ci_lo': cmlo, 'mcc_macro_ci_hi': cmhi,
           'mcc_micro': float(matthews_corrcoef(y_tox.ravel(), yp.ravel())),
           'pr_auc_macro': pr_macro, 'pr_auc_micro': pr_micro}
    for i, l in enumerate(TOX_LABELS):
        row[f'mcc_{l}'] = mcc_pl[i]; row[f'pr_auc_{l}'] = ap[i]
    rows.append(row)

imb_tox = pd.DataFrame(rows)
imb_tox.to_csv(REPORTS / 'eval_imbalance_toxicity.csv', index=False)
run_log.add_output(REPORTS / 'eval_imbalance_toxicity.csv')
print(imb_tox.to_string(index=False))

c:\Python310\lib\site-packages\sklearn\metrics\_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
c:\Python310\lib\site-packages\sklearn\metrics\_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
c:\Python310\lib\site-packages\sklearn\metrics\_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
c:\Python310\lib\site-packages\sklearn\metrics\_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all kno

     model  n_test  mcc_macro  mcc_macro_ci_lo  mcc_macro_ci_hi  mcc_micro  pr_auc_macro  pr_auc_micro  mcc_toxic  pr_auc_toxic  mcc_severe_toxic  pr_auc_severe_toxic  mcc_obscene  pr_auc_obscene  mcc_threat  pr_auc_threat  mcc_insult  pr_auc_insult  mcc_identity_hate  pr_auc_identity_hate
      bert    2392   0.089890         0.014032         0.156189   0.123082      0.061152      0.035428   0.067862      0.047068               0.0                  NaN     0.215287        0.060991         0.0            NaN    0.256194       0.075398                0.0                   NaN
   roberta    2392   0.078930         0.015349         0.142881   0.139047      0.101181      0.059237   0.113420      0.067796               0.0                  NaN     0.263837        0.142459         0.0            NaN    0.096323       0.093287                0.0                   NaN
distilbert    2392   0.082370         0.006014         0.154917   0.136752      0.067169      0.043023   0.101542      0.043513

c:\Python310\lib\site-packages\sklearn\metrics\_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
c:\Python310\lib\site-packages\sklearn\metrics\_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
c:\Python310\lib\site-packages\sklearn\metrics\_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
c:\Python310\lib\site-packages\sklearn\metrics\_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all kno

In [5]:
# Sel 5: Ringkasan markdown siap-kutip
lines = ['# Extended Comparative Metrics: Imbalance-Robust (4 Models)\n',
         'From cached gold-test labels + stored inference scores (no re-training / re-inference).\n',
         '## 3-Class Sentiment — Balanced Accuracy + MCC\n']
cols = ['model', 'balanced_accuracy', 'balanced_acc_ci_lo', 'balanced_acc_ci_hi', 'mcc', 'mcc_ci_lo', 'mcc_ci_hi']
lines.append(imb_sent[[c for c in cols if c in imb_sent.columns]].to_markdown(index=False, floatfmt='.3f'))
lines.append('\n_MCC<0 (CI excluding 0) = worse-than-chance agreement. Balanced acc ~0.33 = random for 3 classes._\n')
lines.append('\n## Multi-Label Toxicity — MCC + PR-AUC\n')
cols = ['model', 'mcc_macro', 'mcc_macro_ci_lo', 'mcc_macro_ci_hi', 'mcc_micro', 'pr_auc_macro', 'pr_auc_micro']
lines.append(imb_tox[[c for c in cols if c in imb_tox.columns]].to_markdown(index=False, floatfmt='.3f'))
lines.append('\n_PR-AUC macro over labels with positive support only (severe_toxic & threat excluded). Threshold-free; complements F1 / Hamming._\n')
(REPORTS / 'comparison_metrics_extended.md').write_text('\n'.join(lines), encoding='utf-8')
run_log.save('reports/run_log.csv')
print('WROTE reports/comparison_metrics_extended.md')

[run_log] 09_imbalance_metrics → 46.8s, 2 outputs, 0 warnings → reports\run_log.csv
WROTE reports/comparison_metrics_extended.md
